In [ ]:
import pandas as pd
import numpy as np
import re
import glob
from pathlib import Path

pattern = r"_embeddings_(.+)$"
validator_folders = ['../age_outputs_validator/', '../rossman_outputs_validator_1/',]
seed_key = lambda p: int(re.findall(r'\d+', Path(p).name)[0])
disp = False

if not disp:
    print(r'\documentclass{article}')
    print(r'\usepackage{booktabs}')
    print(r'\usepackage{amsmath}')
    print(r'\begin{document}')


for validator_folder in validator_folders:
    all_trails = glob.glob(f'{validator_folder}/*')
    for a in all_trails:
        trail = Path(a).name
        trail_escaped = trail.replace('_', r'\_')
        
        if not disp:
            print(f'\n\\section{{{trail_escaped}}}')
        else:
            print(f'# trail name: {trail}')
    
        embd_paths = glob.glob(f'{validator_folder}/{trail}/*')
        rows = []
        for embd_path in embd_paths:
            method_name = re.search(pattern, embd_path).group(1)
            csv_paths = glob.glob(f"{embd_path}/*.csv")
            csvs = [pd.read_csv(c).set_index('metric') for c in sorted(csv_paths, key=seed_key)]
            for df in csvs:
                for i, r in df.iterrows():
                    if '____' in i:
                        _, label, model, main_metric = i.split('____')
                        task = label.split('__')[1] if '__' in label else label
                        if 'f1' not in main_metric:
                            rows.append((method_name, task, model, main_metric, r['value']))
                    else:
                        parts = i.split('__')
                        if len(parts) >= 2:
                            task = parts[1]
                            main_metric = parts[-1]
                            if 'f1' not in main_metric:
                                rows.append((method_name, task, 'best', main_metric, r['value']))
    
        if not rows:
            continue
    
        df = pd.DataFrame(rows, columns=['method', 'task', 'model', 'metric', 'value'])
        grouped = df.groupby(['method', 'task', 'model', 'metric'])['value']
        stats = grouped.agg(mean='mean', std='std').reset_index()
    
        wide_mean = stats.pivot_table(index=['method', 'model'], columns=['task', 'metric'], values='mean')
        wide_std  = stats.pivot_table(index=['method', 'model'], columns=['task', 'metric'], values='std')
    
        best_stats = stats[stats['model'] == 'best']
        if not best_stats.empty:
            best_mean = best_stats.pivot_table(index=['method'], columns=['task','metric'], values='mean')
            best_std  = best_stats.pivot_table(index=['method'], columns=['task','metric'], values='std')
            for idx in wide_mean.index:
                method = idx[0]
                if method in best_mean.index:
                    col_mask = wide_mean.loc[idx].isna()
                    if col_mask.any():
                        for col in wide_mean.columns[col_mask]:
                            if col in best_mean.columns:
                                wide_mean.at[idx, col] = best_mean.at[method, col]
                                wide_std.at[idx, col] = 0.0
    
        wide_display = wide_mean.copy().astype(str)
        for col in wide_display.columns:
            wide_display[col] = (wide_mean[col].map('{:.3f}'.format) +
                                 '±' +
                                 wide_std[col].map('{:.3f}'.format))
    
        def bold_max_mean(s, mean_df):
            if s.name in mean_df.columns:
                mean_col = mean_df[s.name]
                max_val = mean_col.max()
                return ['font-weight: bold' if mean_col.loc[i] == max_val else '' for i in s.index]
            return ['' for _ in s]
    
        for model_key, model_label in [('mlp_torch', 'mlp'), ('lgbm', 'lgbm')]:
            if model_key not in wide_mean.index.get_level_values('model'):
                continue
            display_part = wide_display.xs(model_key, level='model')
            mean_part = wide_mean.xs(model_key, level='model')
            display_part.index.name = model_label
    
            styled = (display_part.style
                      .apply(lambda s: bold_max_mean(s, mean_part), axis=0)
                      .format('{}'))
    
            if disp:
                display(styled)
            else:
                latex_str = styled.to_latex()
                # Fix bold command: \font-weightbold value -> \textbf{value}
                latex_str = re.sub(r'\\font-weightbold\s+([^\s\\]+)', r'\\textbf{\1}', latex_str)
                # Escape underscores
                latex_str = latex_str.replace('_', r'\_')
                # Remove stray braces
                latex_str = latex_str.replace(r'\{', '{').replace(r'\}', '}')
    
                # Grid style with vertical lines, top & bottom hline
                col_match = re.search(r'\\begin{tabular}\{(.*?)\}', latex_str)
                if col_match:
                    ncols = col_match.group(1).count('l') + col_match.group(1).count('c') + col_match.group(1).count('r')
                    new_col_spec = '|' + '|'.join(['l'] * ncols) + '|'
                    latex_str = latex_str.replace(col_match.group(1), new_col_spec)
    
                # Replace booktabs rules
                latex_str = latex_str.replace(r'\toprule', r'\hline')
                latex_str = latex_str.replace(r'\midrule', '')
                latex_str = latex_str.replace(r'\bottomrule', r'\hline')
                latex_str = re.sub(r'\\cmidrule.*?\n', '', latex_str)
    
                # Ensure \hline right after \begin{tabular}
                latex_str = re.sub(
                    r'(\\begin\{tabular\}\{[^}]*\})',
                    r'\1\n\\hline',
                    latex_str
                )
    
                # Force a single \hline just before \end{tabular}
                latex_str = re.sub(r'(\\hline\s*)?(\\end\{tabular\})', r'\\hline\n\2', latex_str)
    
                # Clean up multiple hlines
                latex_str = re.sub(r'(\\hline\s*){2,}', r'\\hline\n', latex_str)
    
                # Remove extra blank lines
                latex_str = re.sub(r'\n\s*\n', '\n', latex_str)
    
                print(r'\noindent ' + latex_str)
                print(r'\vspace{1em}')

if not disp:
    print(r'\end{document}')